# Caso 2 – Terremoto en Colombia: coordenadas y matriz de distancias por carretera

**Objetivo.** Obtener las coordenadas de los 29 puntos de demanda y los 18 sitios (17 candidatos + La Tebaida como reserva) y calcular la matriz de distancias **por carretera** con OSRM (datos de OpenStreetMap). Luego valida la matriz, verifica la restricción de 180 km y calcula el costo de transporte por persona.

**Entradas (misma carpeta):** `demanda_terremoto.csv`, `candidatos_costos.csv`.
**Salidas:** `coordenadas.csv`, `distancias_km.csv` (ancha), `distancias_largo.csv`, `resumen_alcance.csv`.

**Cómo correrlo:** en Colab o Jupyter con internet. Sube los dos CSV y ejecuta todo de arriba abajo. Las distancias actuales del proyecto son *preliminares* hasta que esta validación se ejecute y se revise.

In [ ]:
import re, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

DATA = Path('.')
OSRM = 'https://router.project-osrm.org'
D_MAX_KM = 180          # enunciado
TARIFA = 500            # COP por km-persona (enunciado)
COORD_URLS = [
    'https://datosdelterremoto.org/data/public/divipola_coords.json',   # copia de DIVIPOLA (DANE)
    'https://www.datos.gov.co/resource/gdxc-w37w.json?$limit=5000',      # fuente primaria (datos.gov.co)
]
HEADERS = {'User-Agent': 'CursoDiseno-CadenaSuministro-LaSabana/1.0 (trabajo academico)'}
BBOX = dict(lat=(1.5, 7.0), lon=(-78.5, -74.0))   # caja aproximada Valle + Eje Cafetero

## 1. Cargar nodos (demanda y candidatos)

In [ ]:
dem = pd.read_csv(DATA/'demanda_terremoto.csv', dtype={'divipola': str})
cand = pd.read_csv(DATA/'candidatos_costos.csv', dtype={'divipola': str})
dem['divipola'] = dem.divipola.str.zfill(5); cand['divipola'] = cand.divipola.str.zfill(5)

nodos = pd.concat([
    dem[['divipola','municipio','departamento']].assign(tipo='demanda', id=lambda d: 'D'+d.divipola),
    cand[['divipola','municipio','departamento','id','rol']].rename(columns={'rol':'rol_cand'}).assign(tipo='candidato'),
], ignore_index=True)
assert nodos.divipola.is_unique, 'Un municipio no puede ser a la vez demanda y candidato'
print(len(dem), 'puntos de demanda |', len(cand), 'sitios candidatos (incl. reserva)')

## 2. Coordenadas (DIVIPOLA – cabeceras municipales)
Se intenta primero la copia de DIVIPOLA y luego datos.gov.co. Si ambas fallan, se usa el geocodificador Nominatim (OpenStreetMap) como último recurso. La columna `fuente_coord` deja registro de cuál se usó.

In [ ]:
def _a_df(obj):
    if isinstance(obj, dict):
        listas = [v for v in obj.values() if isinstance(v, list)]
        if listas: return pd.json_normalize(max(listas, key=len))
        if all(isinstance(v, dict) for v in obj.values()):
            return pd.DataFrame.from_dict(obj, orient='index').rename_axis('_clave').reset_index()
        return pd.json_normalize(obj)
    return pd.json_normalize(obj)

def _num(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False), errors='coerce')

def leer_coords(url):
    r = requests.get(url, headers=HEADERS, timeout=60); r.raise_for_status()
    df = _a_df(r.json()); cols = {c.lower(): c for c in df.columns}
    c_lat = next(cols[c] for c in cols if re.match(r'^(lat)', c))
    c_lon = next(cols[c] for c in cols if re.match(r'^(lon|lng)', c))
    c_cod = next((cols[c] for c in cols if re.search(r'(divipola|cod.*mpio|cod.*mun|^codigo$|^code$|^_clave$)', c)), None)
    c_tipo = next((cols[c] for c in cols if re.search(r'tipo', c)), None)
    if c_tipo is not None and df[c_tipo].astype(str).str.contains('CABECERA', case=False).any():
        df = df[df[c_tipo].astype(str).str.contains('CABECERA', case=False)]
    out = pd.DataFrame({'divipola': df[c_cod].astype(str).str.replace(r'\D', '', regex=True).str.zfill(5),
                        'lat': _num(df[c_lat]), 'lon': _num(df[c_lon])}).dropna()
    return out.drop_duplicates('divipola')

def geocodificar_nominatim(fila):
    q = f"{fila.municipio}, {fila.departamento}, Colombia"
    r = requests.get('https://nominatim.openstreetmap.org/search', params={'q': q, 'format': 'json', 'limit': 1},
                     headers=HEADERS, timeout=60); r.raise_for_status(); time.sleep(1.1)   # 1 consulta/segundo
    j = r.json(); return (float(j[0]['lat']), float(j[0]['lon'])) if j else (np.nan, np.nan)

coords, fuente = None, None
for url in COORD_URLS:
    try:
        tabla = leer_coords(url)
        if set(nodos.divipola) <= set(tabla.divipola):
            coords, fuente = tabla, url; break
        print('Faltan códigos en', url, ':', sorted(set(nodos.divipola) - set(tabla.divipola))[:10])
    except Exception as e:
        print('Falló', url, '->', type(e).__name__, e)

if coords is not None:
    nodos = nodos.merge(coords, on='divipola', how='left'); nodos['fuente_coord'] = 'DIVIPOLA: ' + fuente
else:
    print('Usando Nominatim (OpenStreetMap) como respaldo...')
    ll = nodos.apply(geocodificar_nominatim, axis=1, result_type='expand')
    nodos['lat'], nodos['lon'] = ll[0], ll[1]; nodos['fuente_coord'] = 'Nominatim/OSM'

# Validaciones: sin faltantes, dentro de la región, sin duplicados
assert nodos[['lat','lon']].notna().all().all(), nodos[nodos.lat.isna()]
fuera = nodos[~nodos.lat.between(*BBOX['lat']) | ~nodos.lon.between(*BBOX['lon'])]
assert fuera.empty, 'Coordenadas fuera de la región: ' + ', '.join(fuera.municipio)
assert not nodos.duplicated(['lat','lon']).any(), 'Dos municipios con las mismas coordenadas'
nodos.to_csv(DATA/'coordenadas.csv', index=False, encoding='utf-8-sig')
nodos[['id','municipio','tipo','lat','lon','fuente_coord']].head(8)

## 3. Matriz de distancias por carretera con OSRM (Table API)
Una sola consulta con 29 orígenes (demanda) y 18 destinos (sitios). El servicio público admite hasta 100 coordenadas por consulta; aquí hay 47. OSRM devuelve la distancia de la ruta **más rápida** (no necesariamente la más corta), sobre la red de OpenStreetMap, sin cierres posteriores al sismo.

In [ ]:
D = nodos[nodos.tipo == 'demanda'].reset_index(drop=True)
S = nodos[nodos.tipo == 'candidato'].reset_index(drop=True)
todos = pd.concat([D, S], ignore_index=True)
coord_str = ';'.join(f'{lon:.6f},{lat:.6f}' for lon, lat in zip(todos.lon, todos.lat))
src = ';'.join(map(str, range(len(D))))
dst = ';'.join(map(str, range(len(D), len(D) + len(S))))

def osrm_table(intentos=4):
    url = f'{OSRM}/table/v1/driving/{coord_str}'
    params = {'sources': src, 'destinations': dst, 'annotations': 'distance,duration'}
    for k in range(intentos):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=120)
            r.raise_for_status(); j = r.json()
            if j.get('code') == 'Ok': return j
            print('OSRM respondió:', j.get('code'), j.get('message'))
        except Exception as e:
            print(f'Intento {k+1} falló:', type(e).__name__, e)
        time.sleep(3 * (k + 1))
    raise RuntimeError('OSRM no respondió. Reintenta más tarde o usa una instancia propia (docker osrm-backend).')

resp = osrm_table()
dist_km = pd.DataFrame(resp['distances'], index=D.id, columns=S.id, dtype=float) / 1000.0
dur_min = pd.DataFrame(resp['durations'], index=D.id, columns=S.id, dtype=float) / 60.0
snap = pd.Series([w.get('distance') for w in resp['sources'] + resp['destinations']], index=list(D.id) + list(S.id))
print('Matriz', dist_km.shape, '| celdas vacías:', int(dist_km.isna().sum().sum()))
print('Puntos ajustados a la red a más de 3 km del punto original:', list(snap[snap > 3000].index))

## 4. Validación de la matriz

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0088; p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2-p1)/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(np.radians(lon2-lon1)/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

lin = pd.DataFrame(haversine(D.lat.values[:, None], D.lon.values[:, None], S.lat.values[None, :], S.lon.values[None, :]),
                   index=D.id, columns=S.id)
razon = dist_km / lin.replace(0, np.nan)
print('Razón carretera / línea recta:'); print(razon.stack().describe().round(2))

problemas = []
if dist_km.isna().any().any(): problemas.append('Hay pares sin ruta (NaN).')
if (dist_km < lin - 0.5).any().any(): problemas.append('Hay distancias por carretera menores que la línea recta (imposible).')
altos = razon.stack()[razon.stack() > 2.5]
if len(altos): problemas.append(f'{len(altos)} pares con razón > 2,5 (revisar rutas).')
print('\nPROBLEMAS:' if problemas else '\nSin problemas detectados.', *problemas, sep='\n - ')

# Control manual: compara estos pares con Google Maps y anota la diferencia (debería ser < ~10-15 %)
ctrl = [('Pereira','Cartago'), ('Pereira','Chinchiná'), ('Armenia','Cartago'), ('Manizales','Chinchiná'), ('Cali','Palmira'), ('Buenaventura','Palmira')]
nm_d = dict(zip(D.municipio, D.id)); nm_s = dict(zip(S.municipio, S.id))
pd.DataFrame([(a, b, round(dist_km.loc[nm_d[a], nm_s[b]], 1), round(dur_min.loc[nm_d[a], nm_s[b]])) for a, b in ctrl if a in nm_d and b in nm_s],
             columns=['origen','destino','km_OSRM','min_OSRM'])

## 5. Restricción de 180 km y costo de transporte

In [ ]:
a = (dist_km <= D_MAX_KM).astype(int)                   # a_ij = 1 si la distancia permite el traslado
costo_pp = dist_km * TARIFA                              # COP por persona trasladada

base = S[S.rol_cand == 'Candidato'].id.tolist()          # los 17 candidatos base (sin reserva)
por_demanda = pd.DataFrame({'municipio': D.municipio.values,
                            'candidatos_a_180km_base': a[base].sum(axis=1).values,
                            'candidatos_a_180km_con_reserva': a.sum(axis=1).values,
                            'dist_min_km': dist_km.min(axis=1).round(1).values,
                            'candidato_mas_cercano': dist_km.idxmin(axis=1).map(dict(zip(S.id, S.municipio))).values}, index=D.id)
por_sitio = pd.DataFrame({'municipio': S.municipio.values, 'rol': S.rol_cand.values,
                          'demandas_a_180km': a.sum(axis=0).values,
                          'dist_min_km': dist_km.min(axis=0).round(1).values}, index=S.id)
sin_cobertura = por_demanda[por_demanda.candidatos_a_180km_base == 0]
sin_uso = por_sitio[(por_sitio.rol == 'Candidato') & (por_sitio.demandas_a_180km == 0)]
n_validos = int((por_sitio[por_sitio.rol == 'Candidato'].demandas_a_180km > 0).sum())

print(f'Candidatos base válidos (sirven a >=1 punto dentro de {D_MAX_KM} km): {n_validos} de {len(base)}  -> mínimo exigido: 15')
print('Demandas sin ningún candidato base a 180 km:', list(sin_cobertura.municipio) or 'ninguna')
print('Candidatos sin ninguna demanda a 180 km:', list(sin_uso.municipio) or 'ninguno')
por_demanda.sort_values('candidatos_a_180km_base').head(10)

In [ ]:
por_sitio.sort_values('demandas_a_180km')

## 6. Guardar resultados

In [ ]:
dist_km.round(2).to_csv(DATA/'distancias_km.csv', encoding='utf-8-sig')
largo = dist_km.rename_axis(index='id_demanda', columns='id_sitio').stack().rename('km').reset_index()
largo['min'] = dur_min.stack().values; largo['lineal_km'] = lin.stack().values   # mismo orden fila-columna
largo['a_ij'] = (largo.km <= D_MAX_KM).astype(int); largo['costo_transporte_pp'] = largo.km * TARIFA
largo['fuente_distancia'] = f'OSRM {OSRM} ({pd.Timestamp.now():%Y-%m-%d})'
largo.round(3).to_csv(DATA/'distancias_largo.csv', index=False, encoding='utf-8-sig')
pd.concat([por_demanda.assign(nivel='demanda'), por_sitio.assign(nivel='sitio')]).to_csv(DATA/'resumen_alcance.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(6, 7))
ax.scatter(D.lon, D.lat, s=18, label='Demanda'); ax.scatter(S.lon, S.lat, s=40, marker='s', label='Candidatos')
for _, r in todos.iterrows(): ax.annotate(r.municipio, (r.lon, r.lat), fontsize=6, xytext=(2, 2), textcoords='offset points')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud'); ax.legend(); ax.set_title('Nodos del caso'); plt.show()
print('Archivos guardados. Revisa los avisos de la sección 4 antes de cerrar la base.')

## Limitaciones a declarar en el informe
- OSRM usa la red vial de OpenStreetMap **sin cierres ni daños posteriores al sismo**.
- Se usa la **cabecera municipal** como punto representativo; en municipios extensos (Buenaventura, Dagua) las personas pueden estar lejos de ese punto.
- La distancia corresponde a la **ruta más rápida** de OSRM, no necesariamente la más corta.
- El servidor público es de uso ligero; para el resultado final conviene guardar la fecha de consulta (queda en `distancias_largo.csv`).
- Verifica a mano al menos los pares de control de la sección 4 contra Google Maps.